# 00b — Stitch images with shifted beamstops

Load raw or preprocessed images of the same scattering pattern, move the detector mask to each beamstop position, align mutually valid image regions, fit an intensity factor and offset, average every valid contribution, and save one stitched input per polarization for `01_FTH.ipynb`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

BASEFOLDER = Path.cwd().resolve()
sys.path.insert(0, str(BASEFOLDER))
from library.beamstop_stitching import shift_mask, stitch_images
from library.data_loading import Frame, SextantsNexusLoader, load_average
from library.mask_store import MaskStore
print("Base folder:", BASEFOLDER)

## Select the input images

In [ ]:
# Use "preprocessed" for outputs from 01a, or "raw" for NeXus files.
INPUT_KIND = "raw"
IMAGE_IDS = [452, 453, 464, 465, 466, 467, 468, 469, 470, 471]
POLARIZATIONS = ["+", "-", "+", "-", "+", "-", "+", "-", "+", "-"]

# These two lists describe the same images and must have equal lengths.
# Do not combine different scattering states under the same polarization label.
if len(IMAGE_IDS) != len(POLARIZATIONS) or not IMAGE_IDS:
    raise ValueError("IMAGE_IDS and POLARIZATIONS must have the same non-zero length")

RAW_FOLDER = Path("/nfs/ruche/sextants-soleil/com-sextants/COMET_20260902_Cocoons_Laser/")
PREPROCESSED_FOLDER = BASEFOLDER / "processed" / "cleaned_acquisitions"
USER = "rb"
DARK_IDS = 443  # Used only when INPUT_KIND is "raw".

loader = SextantsNexusLoader(RAW_FOLDER)
dark = None
if INPUT_KIND == "raw":
    dark = load_average(loader, DARK_IDS).image
elif INPUT_KIND != "preprocessed":
    raise ValueError('INPUT_KIND must be "raw" or "preprocessed"')

## Load every image separately

In [ ]:
frames = []
energies_eV = []

for image_id in IMAGE_IDS:
    if INPUT_KIND == "raw":
        loaded = loader.load(image_id)
        image = np.asarray(loaded.image, dtype=float) - np.asarray(dark, dtype=float)
        exposure = float(loaded.exposure)
        energy_eV = float(loaded.metadata["energy_eV"])
        source = loaded.source
    else:
        source = PREPROCESSED_FOLDER / f"cleaned_ImId_{image_id:04d}_{USER}.npz"
        with np.load(source, allow_pickle=False) as saved:
            if int(saved["image_id"]) != image_id:
                raise ValueError(f"Image ID inside {source} does not match its filename")
            image = np.asarray(saved["image"], dtype=float)
            energy_eV = float(saved["energy_eV"])
        exposure = 1.0

    frames.append(Frame(str(image_id), image, exposure, Path(source)))
    energies_eV.append(energy_eV)
    print(f"Loaded ID {image_id}: {image.shape} from {source}")

energy_eV = float(np.mean(energies_eV))
print("Photon-energy range:", min(energies_eV), "to", max(energies_eV), "eV")

## Move the detector mask to each beamstop position

Mask shifts use `(row_shift, column_shift)` or `(vertical, horizontal)` pixels. Positive values move down and right.

In [ ]:
BASE_MASK_ID = 95
MASK_SHIFTS = [
    (0, 0), (0, 0),
    (52, -5), (52, -5),
    (-52, -5), (-52, -5),
    (0, -20), (0, -20),
    (-10, 5), (-10, 5),
]

if len(MASK_SHIFTS) != len(IMAGE_IDS):
    raise ValueError("Provide one MASK_SHIFTS entry for every image")

mask_store = MaskStore(BASEFOLDER / "processed" / "mask_pixels")
base_mask = mask_store.load(BASE_MASK_ID, frames[0].image.shape)
shifted_masks = [shift_mask(base_mask, shift) for shift in MASK_SHIFTS]

fig, axes = plt.subplots(1, len(shifted_masks), figsize=(5 * len(shifted_masks), 4), squeeze=False)
for axis, image_id, shift, mask in zip(axes.flat, IMAGE_IDS, MASK_SHIFTS, shifted_masks):
    axis.imshow(mask, cmap="gray", vmin=0, vmax=1)
    axis.set_title(f"ID {image_id}: mask shift {shift}")
    axis.set_axis_off()
plt.tight_layout()
plt.show()

## Find image shifts, factors, and offsets

Registration and intensity fitting use only mutually valid pixels outside the shifted masks. Each polarization is processed independently.

In [ ]:
FIND_IMAGE_SHIFTS = True
MAX_IMAGE_SHIFT = 10.0
REGISTRATION_UPSAMPLE = 10
FIT_ORDER = 1  # 1 = factor + offset; 2 or higher = nonlinear polynomial.
FIT_PERCENTILES = (2, 98)
# Use only this central region to estimate shifts, factors, and offsets.
# Set ALIGNMENT_ROI = None to use the complete mutually valid area.
ALIGNMENT_ROI = np.s_[800:-800, 800:-800]

polarization_labels = []
for label in POLARIZATIONS:
    if label not in polarization_labels:
        polarization_labels.append(label)

stitch_results = []
stitched_image_ids = []
for label in polarization_labels:
    selected = [index for index, value in enumerate(POLARIZATIONS) if value == label]
    selected_frames = [frames[index] for index in selected]
    selected_masks = [shifted_masks[index] for index in selected]
    result = stitch_images(
        selected_frames, selected_masks,
        register=FIND_IMAGE_SHIFTS, max_shift=MAX_IMAGE_SHIFT,
        upsample_factor=REGISTRATION_UPSAMPLE,
        fit_intensity=FIT_ORDER is not None, fit_degree=FIT_ORDER or 1,
        fit_percentiles=FIT_PERCENTILES,
        estimation_roi=ALIGNMENT_ROI,
        use_master_where_valid=True,
    )
    stitch_results.append(result)
    stitched_image_ids.append([IMAGE_IDS[index] for index in selected])

    print(f"Polarization {label}")
    for item in result.prepared_frames:
        print(f"  ID {item.image_id}: image shift={item.shift}, "
              f"polynomial coefficients={item.coefficients}")

## Check alignment and valid overlap

In [ ]:
DISPLAY_PERCENTILES = (1, 99.9)

for label, result in zip(polarization_labels, stitch_results):
    reference = result.prepared_frames[0]
    rows = len(result.prepared_frames)
    estimation_region = np.ones(reference.image.shape, dtype=bool)
    if ALIGNMENT_ROI is not None:
        estimation_region[:] = False
        estimation_region[ALIGNMENT_ROI] = True
    fig, axes = plt.subplots(rows, 4, figsize=(20, 4 * rows), squeeze=False)
    for row, item in enumerate(result.prepared_frames):
        overlap = reference.valid & item.valid
        fit_overlap = overlap & estimation_region
        difference = np.where(overlap, item.image - reference.image, np.nan)
        values = item.image[item.valid & np.isfinite(item.image)]
        vmin, vmax = np.percentile(values, DISPLAY_PERCENTILES)
        residual_limit = np.nanpercentile(np.abs(difference), 99) if overlap.any() else 1
        axes[row, 0].imshow(item.image, cmap="viridis", vmin=vmin, vmax=vmax)
        axes[row, 0].set_title(f"{label} ID {item.image_id}: aligned and calibrated")
        axes[row, 1].imshow(item.valid, cmap="gray", vmin=0, vmax=1)
        axes[row, 1].set_title("valid pixels after mask and alignment")
        axes[row, 2].imshow(difference, cmap="coolwarm", vmin=-residual_limit, vmax=residual_limit)
        axes[row, 2].set_title("difference from reference in valid overlap")

        # Scatter points and the polynomial used to map this image to the master.
        moving_values = item.fit_input[fit_overlap]
        reference_values = reference.image[fit_overlap]
        low, high = np.percentile(moving_values, FIT_PERCENTILES)
        fitted = (moving_values >= low) & (moving_values <= high)
        plot_step = max(1, fitted.sum() // 5000)
        axes[row, 3].scatter(
            moving_values[fitted][::plot_step], reference_values[fitted][::plot_step],
            s=3, alpha=0.2, color="tab:blue", label="fit pixels",
        )
        fit_x = np.linspace(low, high, 300)
        axes[row, 3].plot(
            fit_x, np.polyval(item.coefficients, fit_x),
            color="red", linewidth=2, label=f"order {len(item.coefficients) - 1} fit",
        )
        axes[row, 3].set_title("master intensity vs input intensity")
        axes[row, 3].set_xlabel("input intensity")
        axes[row, 3].set_ylabel("master intensity")
        axes[row, 3].legend()
        for axis in axes[row, :3]:
            axis.set_axis_off()
    plt.tight_layout()
    plt.show()

## Inspect the combined images

In [ ]:
for label, result in zip(polarization_labels, stitch_results):
    finite = result.image[np.isfinite(result.image)]
    vmin, vmax = np.percentile(finite, DISPLAY_PERCENTILES)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    shown = axes[0].imshow(result.image, cmap="viridis", vmin=vmin, vmax=vmax)
    axes[0].set_title(f"{label}: stitched average")
    fig.colorbar(shown, ax=axes[0], label="Intensity")
    axes[1].imshow(result.source_count, cmap="viridis", vmin=0)
    axes[1].set_title("number of contributing images")
    axes[2].imshow(result.missing_mask, cmap="gray", vmin=0, vmax=1)
    axes[2].set_title("final mask (white = missing)")
    for axis in axes:
        axis.set_axis_off()
    plt.tight_layout()
    plt.show()

## Save files for 01_FTH.ipynb

In [ ]:
OUTPUT_FOLDER = BASEFOLDER / "processed" / "stitched"
OUTPUT_NAME = "moved_beamstop"
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

output_files = []
for label, image_ids, result in zip(polarization_labels, stitched_image_ids, stitch_results):
    label_name = "plus" if label == "+" else "minus" if label == "-" else str(label)
    output_file = OUTPUT_FOLDER / f"stitched_{OUTPUT_NAME}_{label_name}_{USER}.npz"
    image_shifts = np.asarray([item.shift for item in result.prepared_frames], dtype=float)
    fit_coefficients = np.asarray([item.coefficients for item in result.prepared_frames], dtype=float)
    factors = fit_coefficients[:, 0] if FIT_ORDER == 1 else np.full(len(image_ids), np.nan)
    offsets = fit_coefficients[:, -1]
    selected_mask_shifts = np.asarray([
        MASK_SHIFTS[index] for index, value in enumerate(POLARIZATIONS) if value == label
    ], dtype=float)
    reference_exposure = frames[IMAGE_IDS.index(image_ids[0])].exposure
    np.savez_compressed(
        output_file, image=result.image,
        mask_pixel=result.missing_mask.astype(np.uint8),
        source_count=result.source_count, reference_exposure=float(reference_exposure),
        ordered_ids=np.asarray(image_ids, dtype=int), polarization=np.asarray(label),
        mask_shifts=selected_mask_shifts, image_shifts=image_shifts,
        fit_order=-1 if FIT_ORDER is None else FIT_ORDER,
        fit_coefficients=fit_coefficients, factors=factors, offsets=offsets,
        energy_eV=energy_eV,
    )
    output_files.append(output_file)
    print(f"Saved {label}: {output_file}")

print("Use these paths as stitched_file in the matching 01_FTH hologram inputs:")
for label, output_file in zip(polarization_labels, output_files):
    print(f'    "{label}": "{output_file.relative_to(BASEFOLDER)}"')

In [ ]:
print("im_ids:", IMAGE_IDS)
print("dark_ids:", DARK_IDS if INPUT_KIND == "raw" else None)